# One-step AIG: measurement report
Минимальный набор графиков для 1-step AIG. Основная единица compute — `GMAC / image`; `ideal_routed` является оценкой условного исполнения, а не фактически выполненной стоимостью текущего Python forward.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "net_complexity").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from net_complexity.studies import load_study

In [ ]:
# Единственная обязательная настройка. Можно указать study или один run.
STUDY_DIR = (REPO_ROOT / "outputs/studies/PUT_STUDY_HERE").resolve()

summary_df, history_df = load_study(STUDY_DIR)
print(f"study: {STUDY_DIR}")
print(f"runs: {history_df['run_name'].nunique()}, history: {history_df.shape}")
display(summary_df)

## Helpers
Ячейки ниже пропускают отсутствующие метрики с понятным сообщением. Это особенно важно для entropy/KL: notebook их не реконструирует, если метод их не логировал.

In [ ]:
RUN = None  # None = все runs; либо точное значение history_df['run_name']
plot_df = history_df.copy() if RUN is None else history_df[history_df['run_name'] == RUN].copy()
plot_df = plot_df.sort_values(['run_name', 'epoch'])

def present(columns):
    return [column for column in columns if column in plot_df.columns]

def line_panel(columns, title, ylabel=None, logy=False):
    columns = present(columns)
    if not columns:
        print(f"skip {title}: required columns are absent")
        return None
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for run_name, run_df in plot_df.groupby('run_name', sort=False):
        for column in columns:
            ax.plot(run_df['epoch'], run_df[column], label=f"{run_name} · {column}")
    ax.set(title=title, xlabel='epoch', ylabel=ylabel or title)
    if logy:
        ax.set_yscale('log')
    ax.grid(alpha=.25)
    ax.legend(fontsize=8, ncols=2)
    fig.tight_layout()
    return fig

## Accuracy and losses

In [ ]:
_ = line_panel(['train_accuracy', 'valid_accuracy'], 'Train / valid accuracy', 'accuracy')

loss_groups = {
    'Cross-entropy loss': ['train_ce_loss', 'valid_ce_loss'],
    'Regularization loss': ['train_regularization_loss', 'valid_regularization_loss'],
    'Total loss': ['train_loss', 'valid_loss'],
}
for title, columns in loss_groups.items():
    _ = line_panel(columns, title, 'loss')

## Lambda and open residual blocks

In [ ]:
lambda_columns = present(['model_lambda_coef', 'lambda_coef', 'adaptive_lambda'])
_ = line_panel(lambda_columns[:1], 'Lambda schedule', 'lambda', logy=True)

valid_gate_columns = sorted(
    column for column in plot_df.columns
    if column.startswith('valid_g_prob_') and '.layer' in column
)
if valid_gate_columns:
    open_df = plot_df[['run_name', 'epoch']].copy()
    open_df['expected_open_blocks'] = plot_df[valid_gate_columns].sum(axis=1)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for run_name, run_df in open_df.groupby('run_name', sort=False):
        ax.plot(run_df['epoch'], run_df['expected_open_blocks'], label=run_name)
    ax.set(title='Expected number of open residual blocks', xlabel='epoch', ylabel='open blocks')
    ax.grid(alpha=.25); ax.legend(fontsize=8); fig.tight_layout()
else:
    print('skip open blocks: valid_g_prob_<block> columns are absent')
# Закрытые блоки намеренно не рисуются: N_closed = N_total - N_open.

## Valid gate-probability heatmap

In [ ]:
for run_name, run_df in plot_df.groupby('run_name', sort=False):
    if not valid_gate_columns:
        break
    matrix = run_df.set_index('epoch')[valid_gate_columns].T
    labels = [column.removeprefix('valid_g_prob_').removeprefix('backbone.') for column in matrix.index]
    fig, ax = plt.subplots(figsize=(max(10, len(matrix.columns) * .08), max(5, len(labels) * .28)))
    image = ax.imshow(matrix, aspect='auto', interpolation='nearest', vmin=0, vmax=1, cmap='viridis')
    ax.set(title=f'Valid gate probabilities · {run_name}', xlabel='epoch', ylabel='residual block')
    ax.set_yticks(np.arange(len(labels)), labels=labels, fontsize=8)
    step = max(1, len(matrix.columns) // 12)
    ax.set_xticks(np.arange(0, len(matrix.columns), step), matrix.columns[::step])
    fig.colorbar(image, ax=ax, label='mean gate-open probability')
    fig.tight_layout()

## Final test gate probabilities

In [ ]:
def load_test_gate_probs(run_dir):
    summary_path = Path(run_dir) / 'summary.json'
    if not summary_path.exists():
        return {}
    test = json.loads(summary_path.read_text(encoding='utf-8')).get('test', {})
    return {
        key.removeprefix('test_g_prob_').removeprefix('backbone.'): float(value)
        for key, value in test.items()
        if key.startswith('test_g_prob_') and '.layer' in key
    }

for _, run in summary_df.iterrows():
    gates = load_test_gate_probs(run['run_dir'])
    if not gates:
        print(f"skip final test gates for {run['run_name']}: metrics are absent")
        continue
    gate_series = pd.Series(gates).sort_index()
    fig, ax = plt.subplots(figsize=(max(10, len(gate_series) * .5), 4.5))
    gate_series.plot.bar(ax=ax, color='tab:blue')
    ax.axhline(.5, color='black', ls='--', lw=1, alpha=.6)
    ax.set(title=f"Final test gate probabilities · {run['run_name']}", ylabel='gate-open probability', ylim=(0, 1))
    ax.tick_params(axis='x', labelrotation=60); ax.grid(axis='y', alpha=.25); fig.tight_layout()

## Valid accuracy versus ideal routed GMAC

In [ ]:
accuracy_column = next((column for column in ['valid_accuracy', 'valid_acc'] if column in plot_df), None)
routed_column = next((column for column in ['valid_ideal_routed_gmac_per_image', 'valid_aig_ideal_routed_gmac_per_image'] if column in plot_df), None)
if accuracy_column and routed_column:
    fig, ax = plt.subplots(figsize=(7, 5))
    for run_name, run_df in plot_df.groupby('run_name', sort=False):
        points = ax.scatter(run_df[routed_column], run_df[accuracy_column], c=run_df['epoch'], cmap='viridis', s=28, label=run_name)
    ax.set(title='Validation accuracy vs ideal routed compute', xlabel='ideal routed GMAC / image', ylabel='valid accuracy')
    ax.grid(alpha=.25); ax.legend(fontsize=8); fig.colorbar(points, ax=ax, label='epoch'); fig.tight_layout()
else:
    print('skip accuracy–GMAC: valid accuracy or routed GMAC is absent')

## Accuracy–GMAC for gate thresholds 0.3 / 0.5 / 0.7
Этот график использует только отдельные runs, реально запущенные и оценённые с соответствующим threshold. Он намеренно не подставляет одну accuracy для трёх гипотетических маршрутов.

In [ ]:
threshold_columns = [column for column in plot_df.columns if column.endswith('gate_threshold')]
if threshold_columns and accuracy_column and routed_column:
    threshold_column = threshold_columns[0]
    final_rows = plot_df.sort_values('epoch').groupby('run_name', as_index=False).tail(1).copy()
    final_rows['gate_threshold'] = pd.to_numeric(final_rows[threshold_column], errors='coerce')
    sweep = final_rows[final_rows['gate_threshold'].round(6).isin([.3, .5, .7])].sort_values('gate_threshold')
    if sweep.empty:
        print('skip threshold sweep: no evaluated runs for thresholds 0.3, 0.5, 0.7')
    else:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(sweep[routed_column], sweep[accuracy_column], marker='o')
        for _, row in sweep.iterrows():
            ax.annotate(f"t={row['gate_threshold']:.1f}", (row[routed_column], row[accuracy_column]), xytext=(5, 5), textcoords='offset points')
        ax.set(title='Evaluated threshold sweep', xlabel='ideal routed GMAC / image', ylabel='valid accuracy')
        ax.grid(alpha=.25); fig.tight_layout()
else:
    print('skip threshold sweep: gate_threshold config, accuracy, or routed GMAC is absent')

## Optional entropy / KL diagnostics

In [ ]:
information_columns = [
    column for column in plot_df.columns
    if any(token in column.lower() for token in ('entropy', 'kl_div', 'kld', 'jensen_shannon'))
]
if information_columns:
    _ = line_panel(information_columns, 'Logged entropy / divergence diagnostics')
else:
    print('Entropy/KL plots omitted: these quantities are not logged by the selected runs.')